RANDOM FOREST REGRESSOR MODEL

Michael Owens 

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [11]:
# Bringing in the Data

# read data
df = pd.read_csv('CleanPreprocessedExoplanet.csv')
df = df.drop('Unnamed: 0',axis=1)
print(df.shape)
df.head(3)

(15351, 50)


,sy_snum,sy_pnum,cb_flag,pl_controv_flag,ttv_flag,st_nphot,pl_nespec,glon,glat,dec,...,sy_kepmag,st_mass,soltype_Kepler Project Candidate (q1_q12_koi),soltype_Kepler Project Candidate (q1_q16_koi),soltype_Kepler Project Candidate (q1_q17_dr24_koi),soltype_Kepler Project Candidate (q1_q17_dr25_koi),soltype_Kepler Project Candidate (q1_q8_koi),soltype_Published Candidate,soltype_Published Confirmed,log radius
0,1,1,0,0,0,0,0,166.79660,-28.05348,20.599021,...,11.040,0.963861,False,False,False,False,False,False,True,0.372075
1,1,1,0,0,0,0,0,166.79660,-28.05348,20.599021,...,11.040,0.961000,False,False,False,False,False,False,True,0.348305
2,1,2,0,0,0,0,0,282.19917,63.35883,1.968169,...,11.678,1.018000,False,False,False,False,False,True,False,0.079181


Grid Search to Determine the Number of Estimators and Tree Depth

In [12]:
#Split data & define features/targets
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)
X_train = df_train.drop('log radius',axis=1)
X_test  = df_test.drop('log radius',axis=1)
y_train   = df_train['log radius']
y_test    = df_test['log radius']

In [13]:
grid = {'max_depth' : np.arange(1,20,3),'n_estimators':np.arange(1,2500,250)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(19), 'n_estimators': np.int64(501)}
    Optimal Valid R2 = 0.8367001169250218


Implementing a Heap Map to Assist Grid Search

In [14]:
Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

In [16]:
fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Refined Grid Search

In [20]:
grid = {'max_depth' : np.arange(17,21,1),'n_estimators':np.arange(485,515,5)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1,verbose=2)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Number of Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show()

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(20), 'n_estimators': np.int64(485)}
    Optimal Valid R2 = 0.8428343905252419


Evaluating the Model

In [25]:
#results = pd.DataFrame()
#results['trees'] = grid['n_estimators']
#results['train R2'] = rfrCV.cv_results_['mean_train_score']
#results['valid R2']  = rfrCV.cv_results_['mean_test_score']
#results[['train R2','valid R2']].plot.line()

In [24]:
# test R2
print(f" train R2 {rfrCV.score(X_train,y_train):.3f}")
print(f" test R2 {rfrCV.score(X_test,y_test):.3f}")

 train R2 0.970
 test R2 0.906


MSE